# LLM MCP Tests\n\nThis notebook contains OpenAI/LangChain MCP interaction tests only.

In [1]:
import json
import os
import uuid
from pathlib import Path
import sys

# Ensure project root is importable when running from test/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "test" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
from fastapi.testclient import TestClient
from sqlalchemy import text

from src import logger
from src import db_connection
from src import guard
from src import tools
import mcp_server

# Make sure .env is loaded for notebook tests
load_dotenv()

RESULTS = []

def add_result(name: str, status: str, details: str = ""):
    RESULTS.append({"test": name, "status": status, "details": details})

def mark_ok(name: str, details: str = ""):
    add_result(name, "PASS", details)

def mark_fail(name: str, details: str = ""):
    add_result(name, "FAIL", details)

def mark_skip(name: str, details: str = ""):
    add_result(name, "SKIP", details)

print("Imports and test harness ready")

Imports and test harness ready


In [2]:
# Optional OpenAI test (uses OPENAI_API_KEY from .env)
try:
    from langchain_openai import ChatOpenAI

    api_key = os.getenv("OPENAI_API_KEY", "").strip()
    if not api_key:
        mark_skip("openai_call", "OPENAI_API_KEY not set")
    else:
        model = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
        llm = ChatOpenAI(model=model, api_key=api_key, temperature=0)
        msg = llm.invoke("Reply with exactly: OPENAI_OK")
        content = getattr(msg, "content", str(msg))
        if "OPENAI_OK" in str(content):
            mark_ok("openai_call", str(content))
        else:
            mark_fail("openai_call", f"Unexpected response: {content}")
except Exception as e:
    mark_skip("openai_call", f"Skipped due to runtime/network/provider issue: {e}")

print("OpenAI test completed")

OpenAI test completed


## LLM Prompt -> MCP Tool Calls (LangChain)

This demo sends your prompt to the model. The model can decide to call MCP tools via `/mcp`.

In [3]:
import requests
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI

MCP_BASE_URL = os.getenv("MCP_BASE_URL", "http://127.0.0.1:8000")
MCP_API_KEY = os.getenv("MCP_API_KEY", "").strip()
ROOT_USER_PROMPT = ""
local_client = None

SYSTEM_PROMPT = (
    "You are a SQL assistant. Use MCP tools when useful. "
    "You MUST always return a non-empty final plain-text answer with exactly 3 bullet points: "
    "what you found, key numbers, suggested next query."
)

def _mcp_headers(user_prompt: str | None = None):
    h = {"Content-Type": "application/json"}
    if MCP_API_KEY:
        h["x-api-key"] = MCP_API_KEY
    prompt_val = (user_prompt or ROOT_USER_PROMPT or "").strip()
    if prompt_val:
        h["x-user-prompt"] = prompt_val
    return h

def _try_enable_local_fallback():
    global local_client
    if local_client is None:
        local_client = TestClient(mcp_server.app)

def mcp_rpc(method: str, params: dict | None = None, user_prompt: str | None = None):
    payload = {"jsonrpc": "2.0", "id": str(uuid.uuid4()), "method": method}
    if params is not None:
        payload["params"] = params

    try:
        resp = requests.post(f"{MCP_BASE_URL}/mcp", headers=_mcp_headers(user_prompt), json=payload, timeout=30)
        resp.raise_for_status()
        body = resp.json()
    except Exception:
        _try_enable_local_fallback()
        resp = local_client.post("/mcp", headers=_mcp_headers(user_prompt), json=payload)
        body = resp.json()

    if "error" in body:
        raise RuntimeError(f"MCP error: {body['error']}")
    return body

@tool
def mcp_get_schema() -> dict:
    """Get DB tables, views, and relationships from MCP."""
    return mcp_rpc("tools/call", {"name": "get_schema", "arguments": {"user_prompt": ROOT_USER_PROMPT}})["result"]

@tool
def mcp_execute_readonly_sql(sql: str) -> dict:
    """Execute read-only SQL through MCP."""
    return mcp_rpc("tools/call", {"name": "execute_readonly_sql", "arguments": {"sql": sql, "user_prompt": ROOT_USER_PROMPT}})["result"]

@tool
def mcp_preview_table(table_name: str, schema_name: str = "dbo") -> dict:
    """Preview first 5 rows of a table through MCP."""
    return mcp_rpc("tools/call", {"name": "preview_table", "arguments": {"table_name": table_name, "schema_name": schema_name, "user_prompt": ROOT_USER_PROMPT}})["result"]

@tool
def mcp_download_result(sql: str, file_name: str = "", download_mode: str = "link") -> dict:
    """Run read-only SQL and return CSV as link or base64 through MCP."""
    mode = (download_mode or "link").strip().lower()
    if mode not in {"link", "base64"}:
        raise ValueError("download_mode must be link or base64")
    args = {"sql": sql, "download_mode": mode, "user_prompt": ROOT_USER_PROMPT}
    if file_name and file_name.strip():
        args["file_name"] = file_name
    return mcp_rpc("tools/call", {"name": "download_result", "arguments": args})["result"]

@tool
def mcp_explain_reasoning(question: str, chosen_tables: list[str], sql: str) -> dict:
    """Explain SQL reasoning through MCP."""
    return mcp_rpc("tools/call", {"name": "explain_reasoning", "arguments": {"question": question, "chosen_tables": chosen_tables, "sql": sql, "user_prompt": ROOT_USER_PROMPT}})["result"]

api_key = os.getenv("OPENAI_API_KEY", "").strip()
if not api_key:
    raise ValueError("OPENAI_API_KEY is required in .env")

model_name = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
llm = ChatOpenAI(model=model_name, api_key=api_key, temperature=0)
llm_with_tools = llm.bind_tools([mcp_get_schema, mcp_execute_readonly_sql, mcp_preview_table, mcp_download_result, mcp_explain_reasoning])

# Persistent conversation state across turns
CONVERSATION_MESSAGES = [SystemMessage(content=SYSTEM_PROMPT)]

def run_langchain_turn(user_prompt: str) -> str:
    global ROOT_USER_PROMPT
    ROOT_USER_PROMPT = user_prompt.strip()
    CONVERSATION_MESSAGES.append(HumanMessage(content=ROOT_USER_PROMPT))

    tool_map = {
        "mcp_get_schema": mcp_get_schema,
        "mcp_execute_readonly_sql": mcp_execute_readonly_sql,
        "mcp_preview_table": mcp_preview_table,
        "mcp_download_result": mcp_download_result,
        "mcp_explain_reasoning": mcp_explain_reasoning,
    }

    tool_outputs = []
    final_text = ""

    for step in range(1, 6):
        reply = llm_with_tools.invoke(CONVERSATION_MESSAGES)
        CONVERSATION_MESSAGES.append(reply)
        print(f"\nStep {step} model content:", repr(reply.content))
        print("tool_calls:", reply.tool_calls)

        if not reply.tool_calls:
            final_text = (reply.content or "").strip()
            if final_text:
                break
            CONVERSATION_MESSAGES.append(HumanMessage(content="Return now exactly 3 plain-text bullet points. Do not leave empty."))
            continue

        for tc in reply.tool_calls:
            name = tc["name"]
            args = tc.get("args", {})
            if name not in tool_map:
                tool_output = {"error": f"Unknown tool from model: {name}"}
            else:
                try:
                    tool_output = tool_map[name].invoke(args)
                except Exception as exc:
                    tool_output = {"error": f"tool_execution_failed: {exc}"}
            tool_outputs.append({"name": name, "args": args, "output": tool_output})
            print(f"Tool {name} args={args}")
            print("Tool output preview:", repr(str(tool_output)[:500]))
            CONVERSATION_MESSAGES.append(ToolMessage(content=json.dumps(tool_output, ensure_ascii=False), tool_call_id=tc["id"]))

    if not final_text:
        last_sql = None
        last_rows = None
        for rec in reversed(tool_outputs):
            out = rec.get("output", {})
            if isinstance(out, dict):
                if last_sql is None and out.get("sql"):
                    last_sql = out.get("sql")
                if last_rows is None and out.get("row_count") is not None:
                    last_rows = out.get("row_count")
        fallback_rows = last_rows if last_rows is not None else "n/a"
        fallback_sql = last_sql if last_sql else "SELECT TOP (10) * FROM dbo.Dim_Employees"
        final_text = (
            "- what you found: Schema and/or query results were retrieved through MCP tools.\n"
            f"- key numbers: tool_calls={len(tool_outputs)}, row_count={fallback_rows}.\n"
            f"- suggested next query: {fallback_sql}"
        )

    return final_text

print("LangChain + MCP ready")
print("MCP_BASE_URL:", MCP_BASE_URL)
print("Conversation state initialized. Use run_langchain_turn(prompt).")

LangChain + MCP ready
MCP_BASE_URL: http://127.0.0.1:8000
Conversation state initialized. Use run_langchain_turn(prompt).


In [4]:
# First prompt in conversation
first_prompt = """
You are a SQL assistant.
Use MCP tools if needed.
You MUST always return a non-empty final plain-text answer with exactly 3 bullet points:
- what you found
- key numbers
- suggested next query
Question: Find 5 employee names from the warehouse DB and explain briefly.
"""

final_text = run_langchain_turn(first_prompt)
print("\nFinal model response:")
print(final_text)
print("\nConversation messages count:", len(CONVERSATION_MESSAGES))


Step 1 model content: ''
tool_calls: [{'name': 'mcp_get_schema', 'args': {}, 'id': 'call_HdXrLvyuBDNa5MuGRNGTs07s', 'type': 'tool_call'}]


Tool mcp_get_schema args={}
Tool output preview: "{'content': [{'type': 'json', 'json': {'tables': {'dbo.Dim_Customers': [{'name': 'CustomerSK', 'type': 'int'}, {'name': 'CustomerBK', 'type': 'nchar'}, {'name': 'CustomerName', 'type': 'nvarchar'}, {'name': 'City', 'type': 'nvarchar'}, {'name': 'Region', 'type': 'nvarchar'}, {'name': 'Country', 'type': 'nvarchar'}], 'dbo.Dim_Date': [{'name': 'DateKey', 'type': 'int'}, {'name': 'Date', 'type': 'date'}, {'name': 'Year', 'type': 'int'}, {'name': 'Quarter', 'type': 'int'}, {'name': 'Month', 'type': "



Step 2 model content: ''
tool_calls: [{'name': 'mcp_execute_readonly_sql', 'args': {'sql': 'SELECT TOP 5 FirstName, LastName FROM dbo.Dim_Employees'}, 'id': 'call_oIc0BZJbAKTBFC6WVOY0KWsG', 'type': 'tool_call'}]
Tool mcp_execute_readonly_sql args={'sql': 'SELECT TOP 5 FirstName, LastName FROM dbo.Dim_Employees'}
Tool output preview: "{'content': [{'type': 'json', 'json': {'sql': 'SELECT TOP 5 FirstName, LastName FROM dbo.Dim_Employees', 'rows': [{'FirstName': 'Nancy', 'LastName': 'Davolio'}, {'FirstName': 'Andrew', 'LastName': 'Fuller'}, {'FirstName': 'Janet', 'LastName': 'Leverling'}, {'FirstName': 'Margaret', 'LastName': 'Peacock'}, {'FirstName': 'Steven', 'LastName': 'Buchanan'}], 'row_count': 5, 'agent_think_ms': 40, 'sql_exec_ms': 13}}]}"



Step 3 model content: ''
tool_calls: [{'name': 'mcp_explain_reasoning', 'args': {'question': 'Find 5 employee names from the warehouse DB and explain briefly.', 'chosen_tables': ['dbo.Dim_Employees'], 'sql': 'SELECT TOP 5 FirstName, LastName FROM dbo.Dim_Employees'}, 'id': 'call_NcTfIYYgSrm4SaoaY5exdVz7', 'type': 'tool_call'}]
Tool mcp_explain_reasoning args={'question': 'Find 5 employee names from the warehouse DB and explain briefly.', 'chosen_tables': ['dbo.Dim_Employees'], 'sql': 'SELECT TOP 5 FirstName, LastName FROM dbo.Dim_Employees'}
Tool output preview: "{'content': [{'type': 'json', 'json': {'interpretation': 'Question interpreted as: Find 5 employee names from the warehouse DB and explain briefly.', 'tables_used': ['dbo.Dim_Employees'], 'join_logic': 'No JOINs; single-source query.', 'filters': 'No explicit filters.', 'aggregations': ['none'], 'limit_policy': 'Result size is bounded with TOP per policy and max table complexity.', 'agent_think_ms': 0}}]}"



Step 4 model content: '- **What I found**: I retrieved the names of 5 employees from the warehouse database. The employees are Nancy Davolio, Andrew Fuller, Janet Leverling, Margaret Peacock, and Steven Buchanan.\n- **Key numbers**: The query returned 5 employee names, and the execution time was approximately 13 milliseconds.\n- **Suggested next query**: You might want to explore more details about these employees, such as their titles or hire dates, by running: `SELECT FirstName, LastName, Title, HireDate FROM dbo.Dim_Employees WHERE EmployeeSK IN (SELECT TOP 5 EmployeeSK FROM dbo.Dim_Employees)`.'
tool_calls: []

Final model response:
- **What I found**: I retrieved the names of 5 employees from the warehouse database. The employees are Nancy Davolio, Andrew Fuller, Janet Leverling, Margaret Peacock, and Steven Buchanan.
- **Key numbers**: The query returned 5 employee names, and the execution time was approximately 13 milliseconds.
- **Suggested next query**: You might want to expl

In [5]:
# Next prompt in the SAME conversation
next_prompt = "Now show total income by year from Fact_Sales."
next_text = run_langchain_turn(next_prompt)
print("\nFinal model response (next turn):")
print(next_text)
print("\nConversation messages count:", len(CONVERSATION_MESSAGES))


Step 1 model content: ''
tool_calls: [{'name': 'mcp_execute_readonly_sql', 'args': {'sql': "SELECT YEAR(DATEADD(DAY, DateKey, '1899-12-30')) AS Year, SUM(UnitPrice * Quantity * (1 - Discount)) AS TotalIncome FROM dbo.Fact_Sales GROUP BY YEAR(DATEADD(DAY, DateKey, '1899-12-30')) ORDER BY Year"}, 'id': 'call_LLEedyVB0zYoJZYyvrNeGtnC', 'type': 'tool_call'}]


Tool mcp_execute_readonly_sql args={'sql': "SELECT YEAR(DATEADD(DAY, DateKey, '1899-12-30')) AS Year, SUM(UnitPrice * Quantity * (1 - Discount)) AS TotalIncome FROM dbo.Fact_Sales GROUP BY YEAR(DATEADD(DAY, DateKey, '1899-12-30')) ORDER BY Year"}
Tool output preview: '{\'error\': "tool_execution_failed: MCP error: {\'code\': -32603, \'message\': \'Internal server error\'}"}'



Step 2 model content: ''
tool_calls: [{'name': 'mcp_execute_readonly_sql', 'args': {'sql': "SELECT YEAR(DATEADD(DAY, DateKey, '1899-12-30')) AS Year, SUM(UnitPrice * Quantity * (1 - Discount)) AS TotalIncome FROM dbo.Fact_Sales GROUP BY YEAR(DATEADD(DAY, DateKey, '1899-12-30')) ORDER BY Year"}, 'id': 'call_3fM1HVTBDZKRhQqWWtbox6Ls', 'type': 'tool_call'}]


Tool mcp_execute_readonly_sql args={'sql': "SELECT YEAR(DATEADD(DAY, DateKey, '1899-12-30')) AS Year, SUM(UnitPrice * Quantity * (1 - Discount)) AS TotalIncome FROM dbo.Fact_Sales GROUP BY YEAR(DATEADD(DAY, DateKey, '1899-12-30')) ORDER BY Year"}
Tool output preview: '{\'error\': "tool_execution_failed: MCP error: {\'code\': -32603, \'message\': \'Internal server error\'}"}'



Step 3 model content: ''
tool_calls: [{'name': 'mcp_preview_table', 'args': {'table_name': 'dbo.Fact_Sales'}, 'id': 'call_cgvJZgcTGmKm5eD4kea7pjUK', 'type': 'tool_call'}]


Tool mcp_preview_table args={'table_name': 'dbo.Fact_Sales'}
Tool output preview: '{\'error\': "tool_execution_failed: MCP error: {\'code\': -32602, \'message\': \'Invalid table_name.\'}"}'



Step 4 model content: ''
tool_calls: [{'name': 'mcp_get_schema', 'args': {}, 'id': 'call_oImBe91L2wS0AQ0pgDCv9hBo', 'type': 'tool_call'}]


Tool mcp_get_schema args={}
Tool output preview: "{'content': [{'type': 'json', 'json': {'tables': {'dbo.Dim_Customers': [{'name': 'CustomerSK', 'type': 'int'}, {'name': 'CustomerBK', 'type': 'nchar'}, {'name': 'CustomerName', 'type': 'nvarchar'}, {'name': 'City', 'type': 'nvarchar'}, {'name': 'Region', 'type': 'nvarchar'}, {'name': 'Country', 'type': 'nvarchar'}], 'dbo.Dim_Date': [{'name': 'DateKey', 'type': 'int'}, {'name': 'Date', 'type': 'date'}, {'name': 'Year', 'type': 'int'}, {'name': 'Quarter', 'type': 'int'}, {'name': 'Month', 'type': "



Step 5 model content: ''
tool_calls: [{'name': 'mcp_execute_readonly_sql', 'args': {'sql': "SELECT DATEPART(YEAR, DATEADD(DAY, DateKey, '1899-12-30')) AS Year, SUM(UnitPrice * Quantity * (1 - Discount)) AS TotalIncome FROM dbo.Fact_Sales GROUP BY DATEPART(YEAR, DATEADD(DAY, DateKey, '1899-12-30')) ORDER BY Year"}, 'id': 'call_tdWBUh7EZb27P3tg9pQYadI0', 'type': 'tool_call'}]


Tool mcp_execute_readonly_sql args={'sql': "SELECT DATEPART(YEAR, DATEADD(DAY, DateKey, '1899-12-30')) AS Year, SUM(UnitPrice * Quantity * (1 - Discount)) AS TotalIncome FROM dbo.Fact_Sales GROUP BY DATEPART(YEAR, DATEADD(DAY, DateKey, '1899-12-30')) ORDER BY Year"}
Tool output preview: '{\'error\': "tool_execution_failed: MCP error: {\'code\': -32603, \'message\': \'Internal server error\'}"}'

Final model response (next turn):
- what you found: Schema and/or query results were retrieved through MCP tools.
- key numbers: tool_calls=5, row_count=n/a.
- suggested next query: SELECT TOP (10) * FROM dbo.Dim_Employees

Conversation messages count: 20


In [6]:
import base64

# Demo: get CSV inline (base64) and decode it locally
inline_res = mcp_download_result.invoke({
    "sql": "SELECT TOP (5) name FROM sys.tables",
    "download_mode": "base64",
    "file_name": "tables_inline.csv"
})

print("download_mode:", inline_res.get("download_mode"))
print("row_count:", inline_res.get("row_count"))
print("has_content_base64:", bool(inline_res.get("content_base64")))

b64 = inline_res.get("content_base64", "")
if b64:
    csv_text = base64.b64decode(b64.encode("ascii")).decode("utf-8")
    out_path = Path("logs/downloads/tables_inline_from_base64.csv")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(csv_text, encoding="utf-8")
    print("saved:", str(out_path))
    preview_lines = csv_text.splitlines()[:6]
    print("preview lines:")
    for line in preview_lines:
        print(line)
else:
    print("No base64 content returned")


download_mode: None
row_count: None
has_content_base64: False
No base64 content returned


## OpenAI SDK + MCP (Below LangChain)\n\nThis uses OpenAI SDK directly with MCP tool type (no LangChain tool wrappers).

In [7]:
from openai import OpenAI

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is required in .env")

SDK_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
SDK_MCP_BASE_URL = os.getenv("MCP_BASE_URL", "http://127.0.0.1:8000").rstrip("/")
SDK_MCP_API_KEY = os.getenv("MCP_API_KEY", "").strip()

sdk_client = OpenAI(api_key=OPENAI_API_KEY)

sdk_mcp_tool = {
    "type": "mcp",
    "server_label": "local_sql_mcp",
    "server_url": f"{SDK_MCP_BASE_URL}/mcp",
    "require_approval": "never"
}
if SDK_MCP_API_KEY:
    sdk_mcp_tool["headers"] = {"x-api-key": SDK_MCP_API_KEY}

print("OpenAI SDK MCP setup ready")
print("MODEL:", SDK_MODEL)
print("MCP URL:", sdk_mcp_tool["server_url"])

OpenAI SDK MCP setup ready
MODEL: gpt-4.1-mini
MCP URL: http://127.0.0.1:8000/mcp


In [8]:
# Put your own prompt here
sdk_prompt = "Find 5 employee names from the warehouse DB and explain briefly in 3 bullet points."

sdk_tool = dict(sdk_mcp_tool)
sdk_headers = dict(sdk_tool.get("headers", {}))
sdk_headers["x-user-prompt"] = sdk_prompt
sdk_tool["headers"] = sdk_headers

try:
    sdk_response = sdk_client.responses.create(
        model=SDK_MODEL,
        input=sdk_prompt,
        tools=[sdk_tool]
    )

    print("Response ID:", sdk_response.id)
    print("\nFinal text:")
    print(getattr(sdk_response, "output_text", ""))

    print("\nOutput items (inspect MCP behavior):")
    for idx, item in enumerate(getattr(sdk_response, "output", []) or []):
        item_type = getattr(item, "type", None) if not isinstance(item, dict) else item.get("type")
        print(f"[{idx}] type={item_type}")
        print(item)
except Exception as e:
    print("OpenAI SDK MCP run skipped/failed:", e)


OpenAI SDK MCP run skipped/failed: Error code: 424 - {'error': {'message': "Error retrieving tool list from MCP server: 'local_sql_mcp'. Http status code: 424 (Failed Dependency)", 'type': 'external_connector_error', 'param': 'tools', 'code': 'http_error'}}
